
# EXP01 — Berrar Representation Ablation for Task C

This notebook is the first controlled modeling experiment after the verified data pipeline.

## Research questions

1. Does training-selected Berrar recency improve over the fixed-recency Berrar baseline?
2. Does the 18-feature `homeaway` representation improve over the 6-feature `total` representation?
3. How does compact Berrar history compare with the existing 128-feature StatsBomb pre-match representation?
4. Does adding Berrar history improve the existing StatsBomb representation?

## Experimental rules

- **No data is downloaded or rebuilt here.**
- The existing MD1 and P1 data runs are treated as immutable upstream evidence.
- Berrar recency is read from the frozen P1 run; it is not re-selected.
- All P1 comparisons use the same P1-eligible population.
- The same Random Forest probe configuration is used for every learned representation.
- Imputation is inside the sklearn pipeline and is therefore fit on **training rows only**.
- Representation preference is determined from **validation RPS only**.
- The validation decision and experiment configuration are written to disk **before** test evaluation.
- Test results are report-only and cannot trigger a change to `n`, features, split, or RF configuration.
- Market probabilities are never included in a model feature matrix.

Primary metric: **Ranked Probability Score (RPS)**.  
Secondary metrics: Log-Loss, multiclass Brier score, Accuracy, and top-label ECE.


## 1. Configuration

In [1]:

from pathlib import Path

# Exact Drive root from the current project tree.
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/ml_project/code/data_pipeline")

BASELINE_RUN = DRIVE_PROJECT_ROOT / "results" / "Premier_League_2015_16_DATA_ONLY_FULL_REAL"
P1_RUN = (
    DRIVE_PROJECT_ROOT
    / "extracted"
    / "md1_data_pipeline_code"
    / "colab_results"
    / "EPL_2000_01_2015_16_Berrar_P1_HYBRID_REAL"
)

EXPERIMENT_ROOT = DRIVE_PROJECT_ROOT / "results" / "EXP01_P1_REPRESENTATION_TASK_C"

# The successful P1 run is frozen. This is an integrity check, not a value to optimize.
EXPECTED_SELECTED_N = 28

RESET_EXPERIMENT_OUTPUT = True
RUN_TEST_STAGE = True

# Fixed RF representation probe — identical for every learned feature view.
RF_N_ESTIMATORS = 500
RF_MAX_DEPTH = None
RF_MIN_SAMPLES_LEAF = 3
RF_MAX_FEATURES = "sqrt"
RF_CLASS_WEIGHT = None
RANDOM_SEED = 42

ECE_BINS = 10
BOOTSTRAP_RESAMPLES = 10_000


## 2. Mount Drive and resolve immutable input files

In [2]:

import os
import sys
import shutil
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

BASE_GOLD = BASELINE_RUN / "data" / "gold" / "gold_prematch.parquet"
BASE_FEATURE_DICT = BASELINE_RUN / "data" / "gold" / "feature_dictionary.csv"
BASE_SPLIT = BASELINE_RUN / "data" / "gold" / "split_manifest.csv"
BASE_READINESS = BASELINE_RUN / "audit" / "md1_data_readiness.csv"
BASE_LEAKAGE = BASELINE_RUN / "audit" / "leakage_test_results.csv"

P1_FIXED = P1_RUN / "data" / "gold" / "p1_features_total_fixed.parquet"
P1_TOTAL = P1_RUN / "data" / "gold" / "p1_features_total_selected.parquet"
P1_HOMEAWAY = P1_RUN / "data" / "gold" / "p1_features_homeaway_selected.parquet"
P1_SPLIT = P1_RUN / "data" / "gold" / "p1_split_manifest.csv"
P1_SUMMARY = P1_RUN / "audit" / "p1_run_summary.json"
P1_SELECTED_RECENCY = P1_RUN / "audit" / "p1_selected_recency.json"
P1_READINESS = P1_RUN / "audit" / "p1_readiness.csv"
P1_LEAKAGE = P1_RUN / "audit" / "p1_leakage_tests.csv"

INPUT_FILES = {
    "baseline_gold_prematch": BASE_GOLD,
    "baseline_feature_dictionary": BASE_FEATURE_DICT,
    "baseline_split_manifest": BASE_SPLIT,
    "baseline_readiness": BASE_READINESS,
    "baseline_leakage": BASE_LEAKAGE,
    "p1_total_fixed": P1_FIXED,
    "p1_total_selected": P1_TOTAL,
    "p1_homeaway_selected": P1_HOMEAWAY,
    "p1_split_manifest": P1_SPLIT,
    "p1_run_summary": P1_SUMMARY,
    "p1_selected_recency": P1_SELECTED_RECENCY,
    "p1_readiness": P1_READINESS,
    "p1_leakage": P1_LEAKAGE,
}

missing = {name: str(path) for name, path in INPUT_FILES.items() if not path.exists()}
if missing:
    raise FileNotFoundError(
        "EXP01 cannot start because required upstream artifacts are missing:\n"
        + "\n".join(f"  {name}: {path}" for name, path in missing.items())
    )

if RESET_EXPERIMENT_OUTPUT and EXPERIMENT_ROOT.exists():
    shutil.rmtree(EXPERIMENT_ROOT)

for sub in ("data", "models", "predictions", "metrics", "audit", "plots"):
    (EXPERIMENT_ROOT / sub).mkdir(parents=True, exist_ok=True)

print("Baseline run:", BASELINE_RUN)
print("P1 run:", P1_RUN)
print("Experiment output:", EXPERIMENT_ROOT)
print("Required upstream files detected: PASS")


Mounted at /content/drive
Baseline run: /content/drive/MyDrive/ml_project/code/data_pipeline/results/Premier_League_2015_16_DATA_ONLY_FULL_REAL
P1 run: /content/drive/MyDrive/ml_project/code/data_pipeline/extracted/md1_data_pipeline_code/colab_results/EPL_2000_01_2015_16_Berrar_P1_HYBRID_REAL
Experiment output: /content/drive/MyDrive/ml_project/code/data_pipeline/results/EXP01_P1_REPRESENTATION_TASK_C
Required upstream files detected: PASS


## 3. Experiment implementation

In [3]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence
import hashlib
import json
import platform
import shutil
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

CLASS_ORDER = ("H", "D", "A")
P1_TOTAL_FEATURES = (
    "p1_home_scr_total",
    "p1_home_con_total",
    "p1_home_rank_total",
    "p1_away_scr_total",
    "p1_away_con_total",
    "p1_away_rank_total",
)
P1_HOMEAWAY_FEATURES = (
    *P1_TOTAL_FEATURES,
    "p1_home_scr_home",
    "p1_home_con_home",
    "p1_home_rank_home",
    "p1_away_scr_home",
    "p1_away_con_home",
    "p1_away_rank_home",
    "p1_home_scr_away",
    "p1_home_con_away",
    "p1_home_rank_away",
    "p1_away_scr_away",
    "p1_away_con_away",
    "p1_away_rank_away",
)

@dataclass(frozen=True)
class RFProbeConfig:
    n_estimators: int = 500
    max_depth: int | None = None
    min_samples_leaf: int = 3
    max_features: str = "sqrt"
    class_weight: str | None = None
    random_state: int = 42
    n_jobs: int = -1
    imputer_strategy: str = "median"
    ece_bins: int = 10
    bootstrap_resamples: int = 10000


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def write_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")


def make_rf_pipeline(cfg: RFProbeConfig) -> Pipeline:
    return Pipeline([
        ("imputer", SimpleImputer(strategy=cfg.imputer_strategy)),
        ("rf", RandomForestClassifier(
            n_estimators=cfg.n_estimators,
            max_depth=cfg.max_depth,
            min_samples_leaf=cfg.min_samples_leaf,
            max_features=cfg.max_features,
            class_weight=cfg.class_weight,
            random_state=cfg.random_state,
            n_jobs=cfg.n_jobs,
        )),
    ])


def aligned_predict_proba(model: Pipeline, X: pd.DataFrame, class_order: Sequence[str] = CLASS_ORDER) -> np.ndarray:
    raw = model.predict_proba(X)
    learned = [str(c) for c in model.named_steps["rf"].classes_]
    missing = [c for c in class_order if c not in learned]
    if missing:
        raise AssertionError(f"Training split is missing outcome classes: {missing}; learned={learned}")
    out = np.column_stack([raw[:, learned.index(c)] for c in class_order])
    if not np.all(np.isfinite(out)):
        raise AssertionError("Non-finite probabilities produced")
    if not np.allclose(out.sum(axis=1), 1.0, atol=1e-10):
        raise AssertionError("Predicted probabilities do not sum to 1")
    return out


def one_hot(y: Sequence[str], class_order: Sequence[str] = CLASS_ORDER) -> np.ndarray:
    idx = {c: i for i, c in enumerate(class_order)}
    arr = np.zeros((len(y), len(class_order)), dtype=float)
    for r, label in enumerate(y):
        if label not in idx:
            raise ValueError(f"Unknown class {label!r}")
        arr[r, idx[label]] = 1.0
    return arr


def per_match_rps(y_true: Sequence[str], probs: np.ndarray, class_order: Sequence[str] = CLASS_ORDER) -> np.ndarray:
    y = one_hot(list(y_true), class_order)
    k = len(class_order)
    if probs.shape != y.shape:
        raise ValueError(f"probability shape {probs.shape} != target shape {y.shape}")
    cum_p = np.cumsum(probs, axis=1)[:, :-1]
    cum_y = np.cumsum(y, axis=1)[:, :-1]
    return np.sum((cum_p - cum_y) ** 2, axis=1) / (k - 1)


def multiclass_brier(y_true: Sequence[str], probs: np.ndarray, class_order: Sequence[str] = CLASS_ORDER) -> float:
    y = one_hot(list(y_true), class_order)
    return float(np.mean(np.sum((probs - y) ** 2, axis=1)))


def top_label_ece(y_true: Sequence[str], probs: np.ndarray, class_order: Sequence[str] = CLASS_ORDER, n_bins: int = 10) -> float:
    y_true = np.asarray(list(y_true), dtype=object)
    pred_idx = np.argmax(probs, axis=1)
    pred = np.asarray([class_order[i] for i in pred_idx], dtype=object)
    conf = np.max(probs, axis=1)
    correct = (pred == y_true).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        if i == n_bins - 1:
            mask = (conf >= lo) & (conf <= hi)
        else:
            mask = (conf >= lo) & (conf < hi)
        n = int(mask.sum())
        if n == 0:
            continue
        ece += (n / len(conf)) * abs(float(correct[mask].mean()) - float(conf[mask].mean()))
    return float(ece)


def metrics_row(name: str, y_true: Sequence[str], probs: np.ndarray, cfg: RFProbeConfig) -> dict[str, Any]:
    y_true = list(y_true)
    rps = per_match_rps(y_true, probs)
    pred = [CLASS_ORDER[i] for i in np.argmax(probs, axis=1)]
    return {
        "representation": name,
        "n_matches": len(y_true),
        "rps": float(np.mean(rps)),
        "log_loss": float(-np.mean(np.log(np.clip(probs[np.arange(len(y_true)), [CLASS_ORDER.index(y) for y in y_true]], 1e-15, 1.0)))) ,
        "brier": multiclass_brier(y_true, probs),
        "accuracy": float(accuracy_score(y_true, pred)),
        "ece": top_label_ece(y_true, probs, n_bins=cfg.ece_bins),
    }


def prediction_frame(name: str, manifest: pd.DataFrame, probs: np.ndarray) -> pd.DataFrame:
    if len(manifest) != len(probs):
        raise ValueError("manifest/probability row mismatch")
    out = manifest[["match_id", "split", "label_outcome"]].copy().reset_index(drop=True)
    out["representation"] = name
    out["p_home"] = probs[:, 0]
    out["p_draw"] = probs[:, 1]
    out["p_away"] = probs[:, 2]
    out["predicted_outcome"] = np.asarray(CLASS_ORDER, dtype=object)[np.argmax(probs, axis=1)]
    out["per_match_rps"] = per_match_rps(out["label_outcome"].tolist(), probs)
    return out


def reliability_long(name: str, y_true: Sequence[str], probs: np.ndarray, n_bins: int = 10) -> pd.DataFrame:
    y_true = np.asarray(list(y_true), dtype=object)
    rows: list[dict[str, Any]] = []
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    for j, cls in enumerate(CLASS_ORDER):
        p = probs[:, j]
        obs = (y_true == cls).astype(float)
        for i in range(n_bins):
            lo, hi = edges[i], edges[i + 1]
            mask = (p >= lo) & ((p <= hi) if i == n_bins - 1 else (p < hi))
            rows.append({
                "representation": name,
                "class": cls,
                "bin": i,
                "bin_left": lo,
                "bin_right": hi,
                "count": int(mask.sum()),
                "mean_probability": float(p[mask].mean()) if mask.any() else np.nan,
                "observed_frequency": float(obs[mask].mean()) if mask.any() else np.nan,
            })
    return pd.DataFrame(rows)


def paired_rps_bootstrap(pred_long: pd.DataFrame, comparisons: Sequence[tuple[str, str]], n_resamples: int = 10000, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    rows: list[dict[str, Any]] = []
    for left, right in comparisons:
        l = pred_long[pred_long["representation"] == left][["match_id", "per_match_rps"]].rename(columns={"per_match_rps": "left_rps"})
        r = pred_long[pred_long["representation"] == right][["match_id", "per_match_rps"]].rename(columns={"per_match_rps": "right_rps"})
        m = l.merge(r, on="match_id", how="inner", validate="one_to_one").sort_values("match_id")
        if m.empty:
            raise AssertionError(f"No common rows for bootstrap comparison {left} vs {right}")
        if len(m) != len(l) or len(m) != len(r):
            raise AssertionError(f"Comparison populations differ for {left} vs {right}")
        d = (m["left_rps"] - m["right_rps"]).to_numpy(float)
        n = len(d)
        # batch to avoid an unnecessarily large B x N matrix if B is increased later
        means: list[np.ndarray] = []
        remaining = n_resamples
        while remaining > 0:
            b = min(2000, remaining)
            idx = rng.integers(0, n, size=(b, n))
            means.append(d[idx].mean(axis=1))
            remaining -= b
        boot = np.concatenate(means)
        rows.append({
            "left": left,
            "right": right,
            "n_matches": n,
            "delta_definition": "RPS(left) - RPS(right); negative favors left",
            "mean_delta_rps": float(d.mean()),
            "ci95_low": float(np.quantile(boot, 0.025)),
            "ci95_high": float(np.quantile(boot, 0.975)),
            "bootstrap_resamples": int(n_resamples),
            "bootstrap_seed": int(seed),
            "fraction_bootstrap_delta_lt_0": float(np.mean(boot < 0.0)),
        })
    return pd.DataFrame(rows)


def as_bool_series(s: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(s.dtype):
        return s.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(s.dtype):
        return s.fillna(0).astype(float).ne(0)
    return s.astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y"})


def integration_checks(
    baseline: pd.DataFrame,
    baseline_split: pd.DataFrame,
    baseline_dict: pd.DataFrame,
    p1_fixed: pd.DataFrame,
    p1_total: pd.DataFrame,
    p1_homeaway: pd.DataFrame,
    p1_split: pd.DataFrame,
    p1_summary: Mapping[str, Any],
) -> tuple[pd.DataFrame, list[str], pd.DataFrame]:
    checks: list[dict[str, Any]] = []
    def add(name: str, passed: bool, details: str):
        checks.append({"check": name, "passed": bool(passed), "details": details})

    add("baseline one row per match", baseline["match_id"].is_unique, f"rows={len(baseline)}, unique={baseline['match_id'].nunique()}")
    add("P1 selected one row per match", p1_total["match_id"].is_unique, f"rows={len(p1_total)}, unique={p1_total['match_id'].nunique()}")
    add("P1 homeaway one row per match", p1_homeaway["match_id"].is_unique, f"rows={len(p1_homeaway)}, unique={p1_homeaway['match_id'].nunique()}")
    add("P1 fixed one row per match", p1_fixed["match_id"].is_unique, f"rows={len(p1_fixed)}, unique={p1_fixed['match_id'].nunique()}")

    base_ids = set(baseline["match_id"].astype(int))
    p1_ids = set(p1_total["match_id"].astype(int))
    add("380 target IDs align", len(base_ids) == 380 and len(p1_ids) == 380 and base_ids == p1_ids,
        f"baseline={len(base_ids)}, p1={len(p1_ids)}, intersection={len(base_ids & p1_ids)}, base_only={len(base_ids-p1_ids)}, p1_only={len(p1_ids-base_ids)}")

    # Full split alignment.
    bs = baseline_split[["match_id", "split"]].copy()
    ps = p1_split[["match_id", "split"]].copy()
    merged_split = bs.merge(ps, on="match_id", suffixes=("_baseline", "_p1"), how="outer", indicator=True)
    split_mismatch = merged_split[(merged_split["_merge"] != "both") | (merged_split["split_baseline"] != merged_split["split_p1"])]
    add("split manifests align", split_mismatch.empty, f"mismatches={len(split_mismatch)}")

    # Each Gold table must also agree with its own manifest.
    bg = baseline[["match_id", "split"]].merge(
        baseline_split[["match_id", "split"]], on="match_id", suffixes=("_gold", "_manifest"), validate="one_to_one"
    )
    bg_bad = bg[bg["split_gold"].astype(str) != bg["split_manifest"].astype(str)]
    add("baseline Gold split matches manifest", bg_bad.empty, f"mismatches={len(bg_bad)}")
    pg = p1_total[["match_id", "split"]].merge(
        p1_split[["match_id", "split"]], on="match_id", suffixes=("_gold", "_manifest"), validate="one_to_one"
    )
    pg_bad = pg[pg["split_gold"].astype(str) != pg["split_manifest"].astype(str)]
    add("P1 Gold split matches manifest", pg_bad.empty, f"mismatches={len(pg_bad)}")

    # Cross-pipeline metadata/label alignment.
    left_cols = ["match_id", "home_team_id", "away_team_id", "kickoff", "label_outcome", "label_margin_raw"]
    missing_left = [c for c in left_cols if c not in baseline.columns]
    right_cols = ["match_id", "home_team_id", "away_team_id", "kickoff", "outcome", "goal_margin_raw"]
    missing_right = [c for c in right_cols if c not in p1_total.columns]
    if missing_left or missing_right:
        add("cross-pipeline identity/labels align", False, f"missing baseline={missing_left}; missing p1={missing_right}")
        alignment = pd.DataFrame()
    else:
        alignment = baseline[left_cols].merge(p1_total[right_cols], on="match_id", suffixes=("_baseline", "_p1"), validate="one_to_one")
        alignment["home_id_match"] = alignment["home_team_id_baseline"].astype(int) == alignment["home_team_id_p1"].astype(int)
        alignment["away_id_match"] = alignment["away_team_id_baseline"].astype(int) == alignment["away_team_id_p1"].astype(int)
        alignment["kickoff_match"] = pd.to_datetime(alignment["kickoff_baseline"]) == pd.to_datetime(alignment["kickoff_p1"])
        alignment["outcome_match"] = alignment["label_outcome"].astype(str) == alignment["outcome"].astype(str)
        alignment["margin_match"] = alignment["label_margin_raw"].astype(int) == alignment["goal_margin_raw"].astype(int)
        bad = alignment[~alignment[["home_id_match","away_id_match","kickoff_match","outcome_match","margin_match"]].all(axis=1)]
        add("cross-pipeline identity/labels align", bad.empty, f"mismatches={len(bad)}")

    # Feature contract from dictionary, not hard-coded position slicing.
    fd = baseline_dict.copy()
    mask = (fd["table"].astype(str) == "gold_prematch") & as_bool_series(fd["model_eligible"])
    baseline_features = fd.loc[mask, "column"].astype(str).tolist()
    missing_base_features = [c for c in baseline_features if c not in baseline.columns]
    add("baseline feature dictionary resolves", len(missing_base_features) == 0, f"features={len(baseline_features)}, missing={missing_base_features[:10]}")
    add("baseline feature contract = 128", len(baseline_features) == 128, f"feature_count={len(baseline_features)}")
    add("P1 total contract = 6", all(c in p1_total.columns for c in P1_TOTAL_FEATURES), f"present={sum(c in p1_total.columns for c in P1_TOTAL_FEATURES)}/6")
    add("P1 homeaway contract = 18", all(c in p1_homeaway.columns for c in P1_HOMEAWAY_FEATURES), f"present={sum(c in p1_homeaway.columns for c in P1_HOMEAWAY_FEATURES)}/18")

    selected_n = int(p1_summary.get("selected_n_pearson"))
    fixed_n = int(p1_summary.get("fixed_recency_baseline_n"))
    observed_selected = sorted(pd.to_numeric(p1_total["recency_n"], errors="coerce").dropna().astype(int).unique().tolist())
    observed_homeaway = sorted(pd.to_numeric(p1_homeaway["recency_n"], errors="coerce").dropna().astype(int).unique().tolist())
    observed_fixed = sorted(pd.to_numeric(p1_fixed["recency_n"], errors="coerce").dropna().astype(int).unique().tolist())
    add("selected recency frozen", observed_selected == [selected_n] and observed_homeaway == [selected_n], f"summary={selected_n}, total={observed_selected}, homeaway={observed_homeaway}")
    add("fixed recency frozen", observed_fixed == [fixed_n], f"summary={fixed_n}, fixed={observed_fixed}")

    eligible_total = p1_total[as_bool_series(p1_total["p1_eligible"])].copy()
    eligible_ids = set(eligible_total["match_id"].astype(int))
    eligible_fixed = set(p1_fixed.loc[as_bool_series(p1_fixed["p1_eligible"]), "match_id"].astype(int))
    eligible_homeaway = set(p1_homeaway.loc[as_bool_series(p1_homeaway["p1_eligible"]), "match_id"].astype(int))
    add("P1 eligibility population consistent", eligible_ids == eligible_fixed == eligible_homeaway, f"selected={len(eligible_ids)}, fixed={len(eligible_fixed)}, homeaway={len(eligible_homeaway)}")
    add("P1 common population = 374", len(eligible_ids) == 374, f"eligible={len(eligible_ids)}")

    common_split = baseline_split[baseline_split["match_id"].astype(int).isin(eligible_ids)].copy()
    counts = common_split["split"].value_counts().to_dict()
    add("common split counts = 251/77/46", counts.get("train",0)==251 and counts.get("validation",0)==77 and counts.get("test",0)==46, str(counts))

    # Market baseline must be complete on exact common validation/test rows.
    market_cols = ["market_p_home", "market_p_draw", "market_p_away", "odds_tagged"]
    missing_market = [c for c in market_cols if c not in baseline.columns]
    if missing_market:
        add("market baseline available", False, f"missing={missing_market}")
    else:
        common_eval = baseline[baseline["match_id"].astype(int).isin(eligible_ids)].merge(common_split, on="match_id", suffixes=("", "_manifest"))
        eval_rows = common_eval[common_eval["split_manifest"].isin(["validation", "test"])]
        probs = eval_rows[["market_p_home","market_p_draw","market_p_away"]].to_numpy(float)
        market_ok = bool((eval_rows["odds_tagged"].astype(int)==1).all() and np.isfinite(probs).all() and np.allclose(probs.sum(axis=1),1.0,atol=1e-8))
        add("market baseline available on common validation/test", market_ok, f"rows={len(eval_rows)}, tagged={int(eval_rows['odds_tagged'].sum())}")

    # Feature leakage guard: rely on baseline dictionary plus explicit P1 columns.
    forbidden_baseline = fd.loc[mask & fd["role"].astype(str).isin(["label","baseline_only","identifier","metadata","audit"]), "column"].astype(str).tolist()
    add("baseline model features exclude non-feature roles", len(forbidden_baseline)==0, f"bad={forbidden_baseline[:10]}")

    out = pd.DataFrame(checks)
    return out, baseline_features, alignment


def build_common_tables(
    baseline: pd.DataFrame,
    p1_fixed: pd.DataFrame,
    p1_total: pd.DataFrame,
    p1_homeaway: pd.DataFrame,
    baseline_features: Sequence[str],
) -> tuple[pd.DataFrame, dict[str, list[str]], dict[str, pd.DataFrame]]:
    eligible = p1_total[as_bool_series(p1_total["p1_eligible"])][["match_id"]].copy()
    if eligible["match_id"].duplicated().any():
        raise AssertionError("P1 eligible IDs are duplicated")

    meta_cols = [
        "match_id", "split", "kickoff", "home_team_id", "home_team_name",
        "away_team_id", "away_team_name", "label_outcome", "label_margin_raw", "label_margin",
        "market_p_home", "market_p_draw", "market_p_away", "odds_tagged",
    ]
    missing_meta = [c for c in meta_cols if c not in baseline.columns]
    if missing_meta:
        raise KeyError(f"Baseline Gold table missing required experiment metadata: {missing_meta}")

    base_common = baseline[baseline["match_id"].isin(eligible["match_id"])][meta_cols + list(baseline_features)].copy()
    if len(base_common) != len(eligible):
        raise AssertionError(f"Baseline common row count {len(base_common)} != eligible {len(eligible)}")

    p1f = p1_fixed[["match_id", *P1_TOTAL_FEATURES]].copy()
    p1t = p1_total[["match_id", *P1_TOTAL_FEATURES]].copy()
    p1h = p1_homeaway[["match_id", *P1_HOMEAWAY_FEATURES]].copy()

    # Separate views to avoid accidental duplicate P1 total fields.
    tables = {
        "P1_TOTAL_FIXED": base_common[meta_cols].merge(p1f, on="match_id", validate="one_to_one"),
        "P1_TOTAL_SELECTED": base_common[meta_cols].merge(p1t, on="match_id", validate="one_to_one"),
        "P1_HOMEAWAY_SELECTED": base_common[meta_cols].merge(p1h, on="match_id", validate="one_to_one"),
        "MD1_BASE_COMMON": base_common.copy(),
        "MD1_PLUS_P1_TOTAL": base_common.merge(p1t, on="match_id", validate="one_to_one"),
        "MD1_PLUS_P1_HOMEAWAY": base_common.merge(p1h, on="match_id", validate="one_to_one"),
    }
    feature_sets = {
        "P1_TOTAL_FIXED": list(P1_TOTAL_FEATURES),
        "P1_TOTAL_SELECTED": list(P1_TOTAL_FEATURES),
        "P1_HOMEAWAY_SELECTED": list(P1_HOMEAWAY_FEATURES),
        "MD1_BASE_COMMON": list(baseline_features),
        "MD1_PLUS_P1_TOTAL": [*baseline_features, *P1_TOTAL_FEATURES],
        "MD1_PLUS_P1_HOMEAWAY": [*baseline_features, *P1_HOMEAWAY_FEATURES],
    }
    for name, table in tables.items():
        if table["match_id"].duplicated().any():
            raise AssertionError(f"{name} has duplicate match IDs")
        if len(table) != 374:
            raise AssertionError(f"{name} expected 374 rows; got {len(table)}")
        missing = [c for c in feature_sets[name] if c not in table.columns]
        if missing:
            raise KeyError(f"{name} missing features {missing[:10]}")
    return base_common, feature_sets, tables


def fit_models(tables: Mapping[str, pd.DataFrame], feature_sets: Mapping[str, Sequence[str]], cfg: RFProbeConfig, models_dir: Path | None = None) -> dict[str, Pipeline]:
    arm_order = [
        "P1_TOTAL_FIXED",
        "P1_TOTAL_SELECTED",
        "P1_HOMEAWAY_SELECTED",
        "MD1_BASE_COMMON",
        "MD1_PLUS_P1_TOTAL",
        "MD1_PLUS_P1_HOMEAWAY",
    ]
    models: dict[str, Pipeline] = {}
    for name in arm_order:
        table = tables[name].sort_values(["kickoff", "match_id"]).reset_index(drop=True)
        cols = list(feature_sets[name])
        train = table[table["split"] == "train"].copy()
        if len(train) != 251:
            raise AssertionError(f"{name} train rows {len(train)} != 251")
        model = make_rf_pipeline(cfg)
        model.fit(train[cols], train["label_outcome"].astype(str))
        # Confirm all three classes were learned before any evaluation.
        learned = set(str(c) for c in model.named_steps["rf"].classes_)
        if learned != set(CLASS_ORDER):
            raise AssertionError(f"{name} training classes={sorted(learned)}; expected={list(CLASS_ORDER)}")
        models[name] = model
        if models_dir is not None:
            models_dir.mkdir(parents=True, exist_ok=True)
            joblib.dump(model, models_dir / f"{name}.joblib")
    return models


def evaluate_split(models: Mapping[str, Pipeline], tables: Mapping[str, pd.DataFrame], feature_sets: Mapping[str, Sequence[str]], cfg: RFProbeConfig, split: str):
    expected = 77 if split == "validation" else 46 if split == "test" else None
    if expected is None:
        raise ValueError("evaluate_split is only for validation or test")
    arm_order = [
        "P1_TOTAL_FIXED",
        "P1_TOTAL_SELECTED",
        "P1_HOMEAWAY_SELECTED",
        "MD1_BASE_COMMON",
        "MD1_PLUS_P1_TOTAL",
        "MD1_PLUS_P1_HOMEAWAY",
    ]
    preds: list[pd.DataFrame] = []
    metrics: list[dict[str, Any]] = []
    reliability: list[pd.DataFrame] = []
    for name in arm_order:
        table = tables[name].sort_values(["kickoff", "match_id"]).reset_index(drop=True)
        part = table[table["split"] == split].sort_values(["kickoff", "match_id"]).reset_index(drop=True)
        if len(part) != expected:
            raise AssertionError(f"{name} {split} rows {len(part)} != {expected}")
        cols = list(feature_sets[name])
        probs = aligned_predict_proba(models[name], part[cols])
        preds.append(prediction_frame(name, part, probs))
        metrics.append({**metrics_row(name, part["label_outcome"].astype(str), probs, cfg), "feature_count": len(cols), "model": "RandomForest fixed probe"})
        reliability.append(reliability_long(name, part["label_outcome"].astype(str), probs, cfg.ece_bins).assign(split=split))

    base = tables["MD1_BASE_COMMON"].sort_values(["kickoff", "match_id"]).reset_index(drop=True)
    part = base[base["split"] == split].sort_values(["kickoff", "match_id"]).reset_index(drop=True)
    if not bool((part["odds_tagged"].astype(int) == 1).all()):
        raise AssertionError(f"Market baseline is not tagged for all common {split} matches")
    probs = part[["market_p_home","market_p_draw","market_p_away"]].to_numpy(float)
    if not np.isfinite(probs).all() or not np.allclose(probs.sum(axis=1), 1.0, atol=1e-8):
        raise AssertionError(f"Invalid market probabilities on {split}")
    preds.append(prediction_frame("MARKET", part, probs))
    metrics.append({**metrics_row("MARKET", part["label_outcome"].astype(str), probs, cfg), "feature_count": 0, "model": "De-vigged Football-Data 1X2 market baseline"})
    reliability.append(reliability_long("MARKET", part["label_outcome"].astype(str), probs, cfg.ece_bins).assign(split=split))

    return (
        pd.concat(preds, ignore_index=True),
        pd.DataFrame(metrics).sort_values("rps").reset_index(drop=True),
        pd.concat(reliability, ignore_index=True),
    )


def validation_decisions(val_metrics: pd.DataFrame, selected_n: int, fixed_n: int) -> dict[str, Any]:
    by = val_metrics.set_index("representation")
    p1_shape_candidates = ["P1_TOTAL_SELECTED", "P1_HOMEAWAY_SELECTED"]
    aug_candidates = ["MD1_BASE_COMMON", "MD1_PLUS_P1_TOTAL", "MD1_PLUS_P1_HOMEAWAY"]
    p1_winner = by.loc[p1_shape_candidates, "rps"].idxmin()
    aug_winner = by.loc[aug_candidates, "rps"].idxmin()
    return {
        "selection_basis": "validation RPS only; test metrics not consulted",
        "selected_n_frozen_before_experiment": int(selected_n),
        "fixed_n": int(fixed_n),
        "preferred_p1_shape_by_validation_rps": str(p1_winner),
        "preferred_project_representation_by_validation_rps": str(aug_winner),
        "validation_rps": {name: float(by.loc[name, "rps"]) for name in by.index},
    }


def missingness_table(tables: Mapping[str, pd.DataFrame], feature_sets: Mapping[str, Sequence[str]]) -> pd.DataFrame:
    rows = []
    for name, table in tables.items():
        for split in ("train", "validation", "test"):
            part = table[table["split"] == split]
            cols = list(feature_sets[name])
            total_cells = len(part) * len(cols)
            missing = int(part[cols].isna().sum().sum())
            rows.append({
                "representation": name,
                "split": split,
                "rows": len(part),
                "features": len(cols),
                "missing_cells": missing,
                "missing_fraction": float(missing / total_cells) if total_cells else 0.0,
                "columns_with_missing": int((part[cols].isna().sum() > 0).sum()),
            })
    return pd.DataFrame(rows)


def feature_contracts(feature_sets: Mapping[str, Sequence[str]]) -> pd.DataFrame:
    return pd.DataFrame([{"representation": k, "feature_count": len(v), "features": "|".join(v)} for k,v in feature_sets.items()])


def plot_rps(test_metrics: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    d = test_metrics.sort_values("rps", ascending=True)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(d["representation"], d["rps"])
    ax.set_xlabel("Test Ranked Probability Score (lower is better)")
    ax.set_ylabel("Representation")
    ax.set_title("EXP01 — Task C representation comparison")
    ax.invert_yaxis()
    fig.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)


def plot_reliability(reliability: pd.DataFrame, representation: str, split: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    d = reliability[(reliability["representation"] == representation) & (reliability["split"] == split)]
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0,1],[0,1], linestyle="--", label="ideal")
    for cls in CLASS_ORDER:
        c = d[(d["class"] == cls) & (d["count"] > 0)]
        ax.plot(c["mean_probability"], c["observed_frequency"], marker="o", label=cls)
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed frequency")
    ax.set_title(f"Reliability — {representation} ({split})")
    ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)


def plot_delta(pred_test: pd.DataFrame, left: str, right: str, path: Path) -> None:
    l = pred_test[pred_test["representation"] == left][["match_id","per_match_rps"]].rename(columns={"per_match_rps":"left"})
    r = pred_test[pred_test["representation"] == right][["match_id","per_match_rps"]].rename(columns={"per_match_rps":"right"})
    m = l.merge(r, on="match_id", validate="one_to_one").sort_values("match_id")
    m["delta"] = m["left"] - m["right"]
    fig, ax = plt.subplots(figsize=(10,5))
    ax.axhline(0.0, linestyle="--")
    ax.plot(np.arange(len(m)), m["delta"].to_numpy(), marker="o", linewidth=1)
    ax.set_xlabel("Common test match (chronological index)")
    ax.set_ylabel(f"RPS({left}) - RPS({right})")
    ax.set_title("Per-match RPS delta; negative favors left representation")
    fig.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)


def environment_manifest() -> dict[str, Any]:
    return {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__,
    }


## 4. Metric unit tests and environment capture

In [4]:

import numpy as np
import pandas as pd
from IPython.display import display

cfg = RFProbeConfig(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    min_samples_leaf=RF_MIN_SAMPLES_LEAF,
    max_features=RF_MAX_FEATURES,
    class_weight=RF_CLASS_WEIGHT,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    ece_bins=ECE_BINS,
    bootstrap_resamples=BOOTSTRAP_RESAMPLES,
)

# RPS unit contracts.
assert np.isclose(per_match_rps(["H"], np.array([[1.0, 0.0, 0.0]]))[0], 0.0)
assert np.isclose(per_match_rps(["D"], np.array([[0.0, 1.0, 0.0]]))[0], 0.0)
assert np.isclose(per_match_rps(["A"], np.array([[0.0, 0.0, 1.0]]))[0], 0.0)
assert np.isclose(multiclass_brier(["H"], np.array([[1.0, 0.0, 0.0]])), 0.0)

environment = environment_manifest()
environment["class_order"] = list(CLASS_ORDER)
environment["rf_probe"] = asdict(cfg)
write_json(environment, EXPERIMENT_ROOT / "audit" / "environment_and_probe_config.json")

print("Metric unit tests: PASS")
display(pd.Series(environment, name="value").to_frame())


Metric unit tests: PASS


,value
python,"3.12.13 (main, Mar 4 2026, 09:23:07) [GCC 11...."
platform,Linux-6.6.122+-x86_64-with-glibc2.35
numpy,2.0.2
pandas,2.2.2
scikit_learn,1.6.1
joblib,1.5.3
class_order,"[H, D, A]"
rf_probe,"{'n_estimators': 500, 'max_depth': None, 'min_..."


## 5. Hash and load the frozen upstream artifacts

In [5]:

import json
import pandas as pd

input_manifest_rows = []
for name, path in INPUT_FILES.items():
    input_manifest_rows.append({
        "name": name,
        "path": str(path),
        "bytes": int(path.stat().st_size),
        "sha256": sha256_file(path),
    })
input_manifest = pd.DataFrame(input_manifest_rows)
input_manifest.to_csv(EXPERIMENT_ROOT / "audit" / "input_manifest.csv", index=False)

baseline = pd.read_parquet(BASE_GOLD)
baseline_dict = pd.read_csv(BASE_FEATURE_DICT)
baseline_split = pd.read_csv(BASE_SPLIT)

p1_fixed = pd.read_parquet(P1_FIXED)
p1_total = pd.read_parquet(P1_TOTAL)
p1_homeaway = pd.read_parquet(P1_HOMEAWAY)
p1_split = pd.read_csv(P1_SPLIT)

p1_summary = json.loads(P1_SUMMARY.read_text(encoding="utf-8"))
p1_selected_recency = json.loads(P1_SELECTED_RECENCY.read_text(encoding="utf-8"))

print("Loaded:")
print("  baseline Gold rows:", len(baseline))
print("  P1 fixed rows:", len(p1_fixed))
print("  P1 selected-total rows:", len(p1_total))
print("  P1 selected-homeaway rows:", len(p1_homeaway))
print("  P1 summary selected n:", p1_summary.get("selected_n_pearson"))
print()
display(input_manifest)


Loaded:
  baseline Gold rows: 380
  P1 fixed rows: 380
  P1 selected-total rows: 380
  P1 selected-homeaway rows: 380
  P1 summary selected n: 28



,name,path,bytes,sha256
0,baseline_gold_prematch,/content/drive/MyDrive/ml_project/code/data_pi...,342478,763117965fd2944133b28f89d95e4729b32d71f7771b40...
1,baseline_feature_dictionary,/content/drive/MyDrive/ml_project/code/data_pi...,83782,2d6551e16d8fb3333c2a94ec19b8f8ecb890a4bdb67d12...
2,baseline_split_manifest,/content/drive/MyDrive/ml_project/code/data_pi...,15207,f3439213b68d5d79d54b065992027fafb62fcd858d31f2...
3,baseline_readiness,/content/drive/MyDrive/ml_project/code/data_pi...,1511,8fd9018cea5b5faac6e04e768b4d5c6c93452e7603b243...
4,baseline_leakage,/content/drive/MyDrive/ml_project/code/data_pi...,1467,37f25e33ac9727b39c6970bd7f0baa6eb9f88e59abadcc...
5,p1_total_fixed,/content/drive/MyDrive/ml_project/code/data_pi...,28879,0b30d97fb689cb11acc49ac35930e203c5c5f1f4401a70...
6,p1_total_selected,/content/drive/MyDrive/ml_project/code/data_pi...,29980,941d42742e420d7901ae988489e60d805fe74e6af61a03...
7,p1_homeaway_selected,/content/drive/MyDrive/ml_project/code/data_pi...,46395,9d7bd7e8abb39b127c44c3118517413e1aeca04bc21f17...
8,p1_split_manifest,/content/drive/MyDrive/ml_project/code/data_pi...,15207,f3439213b68d5d79d54b065992027fafb62fcd858d31f2...
9,p1_run_summary,/content/drive/MyDrive/ml_project/code/data_pi...,1354,36bdfec4d3a24e0d8b93328e8ee7b407711ebdd3635f3b...


## 6. Require both upstream pipelines to be ready

In [6]:

def require_passed_audit(path: Path, label: str):
    audit = pd.read_csv(path)
    if "passed" not in audit.columns:
        raise KeyError(f"{label} audit has no 'passed' column: {path}")
    passed = as_bool_series(audit["passed"])
    if not bool(passed.all()):
        bad = audit.loc[~passed]
        display(bad)
        raise AssertionError(f"{label} contains failed checks; EXP01 is blocked.")
    print(f"{label}: {int(passed.sum())}/{len(passed)} PASS")
    return audit

base_ready = require_passed_audit(BASE_READINESS, "MD1 readiness")
base_leak = require_passed_audit(BASE_LEAKAGE, "MD1 leakage/integrity")
p1_ready = require_passed_audit(P1_READINESS, "P1 readiness")
p1_leak = require_passed_audit(P1_LEAKAGE, "P1 leakage/source")

selected_n = int(p1_summary["selected_n_pearson"])
fixed_n = int(p1_summary["fixed_recency_baseline_n"])

if selected_n != EXPECTED_SELECTED_N:
    raise AssertionError(
        f"Frozen P1 selected n changed: expected {EXPECTED_SELECTED_N}, observed {selected_n}"
    )

print("Frozen selected n:", selected_n)
print("Frozen fixed-recency baseline n:", fixed_n)


MD1 readiness: 15/15 PASS
MD1 leakage/integrity: 12/12 PASS
P1 readiness: 15/15 PASS
P1 leakage/source: 13/13 PASS
Frozen selected n: 28
Frozen fixed-recency baseline n: 10


## 7. Cross-pipeline integration gate

In [7]:

checks, baseline_features, alignment = integration_checks(
    baseline=baseline,
    baseline_split=baseline_split,
    baseline_dict=baseline_dict,
    p1_fixed=p1_fixed,
    p1_total=p1_total,
    p1_homeaway=p1_homeaway,
    p1_split=p1_split,
    p1_summary=p1_summary,
)

checks.to_csv(EXPERIMENT_ROOT / "audit" / "integration_checks.csv", index=False)
if not alignment.empty:
    alignment.to_csv(EXPERIMENT_ROOT / "audit" / "cross_pipeline_alignment.csv", index=False)

display(checks)

if not bool(checks["passed"].all()):
    failed = checks.loc[~checks["passed"]]
    display(failed)
    raise AssertionError("EXP01 BLOCKED: cross-pipeline integration gate failed.")

print()
print("Integration gate: PASS")
print("MD1 model-eligible feature count:", len(baseline_features))
print("P1 selected n:", selected_n)


,check,passed,details
0,baseline one row per match,True,"rows=380, unique=380"
1,P1 selected one row per match,True,"rows=380, unique=380"
2,P1 homeaway one row per match,True,"rows=380, unique=380"
3,P1 fixed one row per match,True,"rows=380, unique=380"
4,380 target IDs align,True,"baseline=380, p1=380, intersection=380, base_o..."
5,split manifests align,True,mismatches=0
6,baseline Gold split matches manifest,True,mismatches=0
7,P1 Gold split matches manifest,True,mismatches=0
8,cross-pipeline identity/labels align,True,mismatches=0
9,baseline feature dictionary resolves,True,"features=128, missing=[]"



Integration gate: PASS
MD1 model-eligible feature count: 128
P1 selected n: 28


## 8. Construct the six controlled representation arms

In [8]:

base_common, feature_sets, tables = build_common_tables(
    baseline=baseline,
    p1_fixed=p1_fixed,
    p1_total=p1_total,
    p1_homeaway=p1_homeaway,
    baseline_features=baseline_features,
)

contracts = feature_contracts(feature_sets)
missingness = missingness_table(tables, feature_sets)

contracts.to_csv(EXPERIMENT_ROOT / "audit" / "feature_contracts.csv", index=False)
missingness.to_csv(EXPERIMENT_ROOT / "audit" / "missingness.csv", index=False)

# Canonical common manifest.
manifest_cols = [
    "match_id", "split", "kickoff", "home_team_id", "home_team_name",
    "away_team_id", "away_team_name", "label_outcome", "label_margin_raw",
    "label_margin", "odds_tagged",
]
common_manifest = base_common[manifest_cols].sort_values(["kickoff", "match_id"]).reset_index(drop=True)
common_manifest.to_csv(EXPERIMENT_ROOT / "data" / "common_match_manifest.csv", index=False)

# Save every representation as a modeling-ready table.
file_map = {
    "P1_TOTAL_FIXED": "prematch_p1_total_fixed.parquet",
    "P1_TOTAL_SELECTED": "prematch_p1_total_selected.parquet",
    "P1_HOMEAWAY_SELECTED": "prematch_p1_homeaway_selected.parquet",
    "MD1_BASE_COMMON": "prematch_md1_common.parquet",
    "MD1_PLUS_P1_TOTAL": "prematch_md1_plus_p1_total.parquet",
    "MD1_PLUS_P1_HOMEAWAY": "prematch_md1_plus_p1_homeaway.parquet",
}

for name, filename in file_map.items():
    tables[name].to_parquet(EXPERIMENT_ROOT / "data" / filename, index=False)

# Explicit feature names per arm.
write_json(
    {name: list(cols) for name, cols in feature_sets.items()},
    EXPERIMENT_ROOT / "audit" / "feature_sets.json",
)

print("Representation contracts")
display(contracts[["representation", "feature_count"]])

print("Missingness before train-only median imputation")
display(missingness)

assert len(feature_sets["P1_TOTAL_FIXED"]) == 6
assert len(feature_sets["P1_TOTAL_SELECTED"]) == 6
assert len(feature_sets["P1_HOMEAWAY_SELECTED"]) == 18
assert len(feature_sets["MD1_BASE_COMMON"]) == 128
assert len(feature_sets["MD1_PLUS_P1_TOTAL"]) == 134
assert len(feature_sets["MD1_PLUS_P1_HOMEAWAY"]) == 146

print("Six controlled modeling views exported: PASS")


Representation contracts


,representation,feature_count
0,P1_TOTAL_FIXED,6
1,P1_TOTAL_SELECTED,6
2,P1_HOMEAWAY_SELECTED,18
3,MD1_BASE_COMMON,128
4,MD1_PLUS_P1_TOTAL,134
5,MD1_PLUS_P1_HOMEAWAY,146


Missingness before train-only median imputation


,representation,split,rows,features,missing_cells,missing_fraction,columns_with_missing
0,P1_TOTAL_FIXED,train,251,6,0,0.000000,0
1,P1_TOTAL_FIXED,validation,77,6,0,0.000000,0
2,P1_TOTAL_FIXED,test,46,6,0,0.000000,0
3,P1_TOTAL_SELECTED,train,251,6,0,0.000000,0
4,P1_TOTAL_SELECTED,validation,77,6,0,0.000000,0
5,P1_TOTAL_SELECTED,test,46,6,0,0.000000,0
6,P1_HOMEAWAY_SELECTED,train,251,18,0,0.000000,0
7,P1_HOMEAWAY_SELECTED,validation,77,18,0,0.000000,0
8,P1_HOMEAWAY_SELECTED,test,46,18,0,0.000000,0
9,MD1_BASE_COMMON,train,251,128,1080,0.033616,120


Six controlled modeling views exported: PASS



## 9. Fit the fixed RF probe — training split only

No representation-specific tuning occurs here. Each arm gets the same estimator configuration:

- 500 trees
- `max_depth=None`
- `min_samples_leaf=3`
- `max_features="sqrt"`
- no class weighting
- median imputation inside the pipeline
- fixed random seed

The purpose is to probe the **representation**, not optimize Random Forest.


In [9]:

models = fit_models(
    tables=tables,
    feature_sets=feature_sets,
    cfg=cfg,
    models_dir=EXPERIMENT_ROOT / "models",
)

print("Models fit on common TRAIN population only:", 251)
print("Learned representations:", list(models))
print("Training stage: PASS")


Models fit on common TRAIN population only: 251
Learned representations: ['P1_TOTAL_FIXED', 'P1_TOTAL_SELECTED', 'P1_HOMEAWAY_SELECTED', 'MD1_BASE_COMMON', 'MD1_PLUS_P1_TOTAL', 'MD1_PLUS_P1_HOMEAWAY']
Training stage: PASS


## 10. Validation evaluation and representation decision

In [10]:

val_predictions, val_metrics, val_reliability = evaluate_split(
    models=models,
    tables=tables,
    feature_sets=feature_sets,
    cfg=cfg,
    split="validation",
)

val_predictions.to_csv(
    EXPERIMENT_ROOT / "predictions" / "validation_predictions.csv", index=False
)
val_metrics.to_csv(
    EXPERIMENT_ROOT / "metrics" / "validation_metrics.csv", index=False
)
val_reliability.to_csv(
    EXPERIMENT_ROOT / "metrics" / "validation_reliability_bins.csv", index=False
)

decision = validation_decisions(
    val_metrics=val_metrics,
    selected_n=selected_n,
    fixed_n=fixed_n,
)
write_json(
    decision,
    EXPERIMENT_ROOT / "audit" / "representation_decision_validation.json",
)

print("VALIDATION RESULTS — lower RPS is better")
display(val_metrics)

print("Validation-only decisions")
display(pd.Series(decision, name="value").to_frame())


VALIDATION RESULTS — lower RPS is better


,representation,n_matches,rps,log_loss,brier,accuracy,ece,feature_count,model
0,MARKET,77,0.196931,0.974603,0.580026,0.545455,0.060403,0,De-vigged Football-Data 1X2 market baseline
1,P1_TOTAL_SELECTED,77,0.212926,1.039698,0.618526,0.441558,0.107637,6,RandomForest fixed probe
2,P1_HOMEAWAY_SELECTED,77,0.213131,1.038484,0.619191,0.467532,0.069547,18,RandomForest fixed probe
3,MD1_PLUS_P1_HOMEAWAY,77,0.217547,1.048381,0.626288,0.467532,0.078942,146,RandomForest fixed probe
4,MD1_PLUS_P1_TOTAL,77,0.218368,1.041537,0.622437,0.467532,0.103201,134,RandomForest fixed probe
5,P1_TOTAL_FIXED,77,0.223466,1.051925,0.634180,0.506494,0.106890,6,RandomForest fixed probe
6,MD1_BASE_COMMON,77,0.223629,1.064525,0.639151,0.441558,0.036884,128,RandomForest fixed probe


Validation-only decisions


,value
selection_basis,validation RPS only; test metrics not consulted
selected_n_frozen_before_experiment,28
fixed_n,10
preferred_p1_shape_by_validation_rps,P1_TOTAL_SELECTED
preferred_project_representation_by_validation_rps,MD1_PLUS_P1_HOMEAWAY
validation_rps,"{'MARKET': 0.19693130130833358, 'P1_TOTAL_SELE..."



## 11. Freeze the experiment **before test evaluation**

This cell writes the immutable test-opening record. It includes:

- SHA-256 hashes of all upstream artifacts;
- exact common population;
- split counts;
- selected and fixed Berrar recencies;
- class order and metric definitions;
- exact feature sets;
- fixed RF probe configuration;
- validation-only representation decision.

The test stage below is allowed to report results but not alter this record.


In [11]:

common_counts = common_manifest["split"].value_counts().to_dict()

freeze_record = {
    "experiment": "EXP01_P1_REPRESENTATION_TASK_C",
    "test_status_at_write": "NOT_EVALUATED_YET",
    "selection_basis": "validation RPS only",
    "selected_n": selected_n,
    "fixed_n": fixed_n,
    "expected_common_population": 374,
    "common_split_counts": common_counts,
    "class_order": list(CLASS_ORDER),
    "rps_definition": "1/(K-1) * sum over first K-1 cumulative squared probability errors; K=3",
    "brier_definition": "mean over matches of sum_k (p_k - y_k)^2",
    "ece_definition": f"top-label ECE with {ECE_BINS} equal-width confidence bins",
    "rf_probe": asdict(cfg),
    "feature_sets": {name: list(cols) for name, cols in feature_sets.items()},
    "validation_decision": decision,
    "input_files": input_manifest.to_dict("records"),
}

freeze_path = EXPERIMENT_ROOT / "audit" / "FROZEN_BEFORE_TEST.json"
write_json(freeze_record, freeze_path)

if not freeze_path.exists():
    raise AssertionError("Freeze record was not written; test stage is blocked.")

print("FROZEN BEFORE TEST:", freeze_path)
print("Test data has not been used for representation selection.")


FROZEN BEFORE TEST: /content/drive/MyDrive/ml_project/code/data_pipeline/results/EXP01_P1_REPRESENTATION_TASK_C/audit/FROZEN_BEFORE_TEST.json
Test data has not been used for representation selection.


## 12. Test evaluation — report only

In [12]:

if not RUN_TEST_STAGE:
    print("RUN_TEST_STAGE=False. Experiment is frozen after validation; test remains unopened.")
else:
    freeze_path = EXPERIMENT_ROOT / "audit" / "FROZEN_BEFORE_TEST.json"
    if not freeze_path.exists():
        raise AssertionError("Missing FROZEN_BEFORE_TEST.json; test stage is blocked.")

    test_predictions, test_metrics, test_reliability = evaluate_split(
        models=models,
        tables=tables,
        feature_sets=feature_sets,
        cfg=cfg,
        split="test",
    )

    test_predictions.to_csv(
        EXPERIMENT_ROOT / "predictions" / "test_predictions.csv", index=False
    )
    test_metrics.to_csv(
        EXPERIMENT_ROOT / "metrics" / "test_metrics.csv", index=False
    )
    test_reliability.to_csv(
        EXPERIMENT_ROOT / "metrics" / "test_reliability_bins.csv", index=False
    )

    print("TEST RESULTS — report only; lower RPS is better")
    display(test_metrics)


TEST RESULTS — report only; lower RPS is better


,representation,n_matches,rps,log_loss,brier,accuracy,ece,feature_count,model
0,MARKET,46,0.174848,0.961138,0.573607,0.521739,0.141250,0,De-vigged Football-Data 1X2 market baseline
1,MD1_BASE_COMMON,46,0.197268,1.016214,0.605964,0.456522,0.121464,128,RandomForest fixed probe
2,MD1_PLUS_P1_TOTAL,46,0.200219,1.023407,0.612775,0.413043,0.098638,134,RandomForest fixed probe
3,MD1_PLUS_P1_HOMEAWAY,46,0.202090,1.042429,0.622532,0.456522,0.066491,146,RandomForest fixed probe
4,P1_TOTAL_FIXED,46,0.208305,1.046362,0.623790,0.521739,0.095419,6,RandomForest fixed probe
5,P1_HOMEAWAY_SELECTED,46,0.220792,1.133443,0.685315,0.369565,0.136783,18,RandomForest fixed probe
6,P1_TOTAL_SELECTED,46,0.229433,1.167425,0.700290,0.456522,0.168881,6,RandomForest fixed probe


## 13. Paired test-match RPS bootstrap

In [13]:

if RUN_TEST_STAGE:
    comparisons = [
        ("P1_TOTAL_SELECTED", "P1_TOTAL_FIXED"),
        ("P1_HOMEAWAY_SELECTED", "P1_TOTAL_SELECTED"),
        ("P1_HOMEAWAY_SELECTED", "MD1_BASE_COMMON"),
        ("MD1_PLUS_P1_TOTAL", "MD1_BASE_COMMON"),
        ("MD1_PLUS_P1_HOMEAWAY", "MD1_BASE_COMMON"),
        ("MD1_BASE_COMMON", "MARKET"),
        ("MD1_PLUS_P1_TOTAL", "MARKET"),
        ("MD1_PLUS_P1_HOMEAWAY", "MARKET"),
    ]

    bootstrap = paired_rps_bootstrap(
        pred_long=test_predictions,
        comparisons=comparisons,
        n_resamples=BOOTSTRAP_RESAMPLES,
        seed=RANDOM_SEED,
    )
    bootstrap.to_csv(
        EXPERIMENT_ROOT / "metrics" / "paired_rps_bootstrap.csv", index=False
    )

    print("Delta definition: RPS(left) - RPS(right); negative favors LEFT.")
    display(bootstrap)


Delta definition: RPS(left) - RPS(right); negative favors LEFT.


,left,right,n_matches,delta_definition,mean_delta_rps,ci95_low,ci95_high,bootstrap_resamples,bootstrap_seed,fraction_bootstrap_delta_lt_0
0,P1_TOTAL_SELECTED,P1_TOTAL_FIXED,46,RPS(left) - RPS(right); negative favors left,0.021128,-0.012849,0.056051,10000,42,0.1119
1,P1_HOMEAWAY_SELECTED,P1_TOTAL_SELECTED,46,RPS(left) - RPS(right); negative favors left,-0.008641,-0.027178,0.009187,10000,42,0.8233
2,P1_HOMEAWAY_SELECTED,MD1_BASE_COMMON,46,RPS(left) - RPS(right); negative favors left,0.023524,-0.003706,0.049107,10000,42,0.0440
3,MD1_PLUS_P1_TOTAL,MD1_BASE_COMMON,46,RPS(left) - RPS(right); negative favors left,0.002951,-0.003778,0.009444,10000,42,0.1823
4,MD1_PLUS_P1_HOMEAWAY,MD1_BASE_COMMON,46,RPS(left) - RPS(right); negative favors left,0.004822,-0.003589,0.013242,10000,42,0.1321
5,MD1_BASE_COMMON,MARKET,46,RPS(left) - RPS(right); negative favors left,0.022420,-0.003951,0.048824,10000,42,0.0464
6,MD1_PLUS_P1_TOTAL,MARKET,46,RPS(left) - RPS(right); negative favors left,0.025372,0.001179,0.049757,10000,42,0.0198
7,MD1_PLUS_P1_HOMEAWAY,MARKET,46,RPS(left) - RPS(right); negative favors left,0.027242,0.002782,0.051309,10000,42,0.0136


## 14. Plots

In [14]:

if RUN_TEST_STAGE:
    plot_rps(
        test_metrics,
        EXPERIMENT_ROOT / "plots" / "test_rps_comparison.png",
    )

    # Main project comparison.
    plot_delta(
        test_predictions,
        left="MD1_PLUS_P1_HOMEAWAY",
        right="MD1_BASE_COMMON",
        path=EXPERIMENT_ROOT / "plots" / "per_match_rps_delta_md1_plus_p1_homeaway_vs_md1.png",
    )

    # Reliability diagrams are raw/uncalibrated in EXP01.
    for representation in [
        "P1_TOTAL_SELECTED",
        "P1_HOMEAWAY_SELECTED",
        "MD1_BASE_COMMON",
        "MD1_PLUS_P1_TOTAL",
        "MD1_PLUS_P1_HOMEAWAY",
        "MARKET",
    ]:
        plot_reliability(
            test_reliability,
            representation=representation,
            split="test",
            path=EXPERIMENT_ROOT / "plots" / f"reliability_test_{representation}.png",
        )

    print("Plots written to:", EXPERIMENT_ROOT / "plots")


Plots written to: /content/drive/MyDrive/ml_project/code/data_pipeline/results/EXP01_P1_REPRESENTATION_TASK_C/plots


## 15. Final experiment readiness and summary

In [15]:

readiness_rows = [
    {
        "check": "upstream MD1 readiness",
        "passed": bool(as_bool_series(base_ready["passed"]).all()),
        "details": f"{int(as_bool_series(base_ready['passed']).sum())}/{len(base_ready)}",
    },
    {
        "check": "upstream P1 readiness",
        "passed": bool(as_bool_series(p1_ready["passed"]).all()),
        "details": f"{int(as_bool_series(p1_ready['passed']).sum())}/{len(p1_ready)}",
    },
    {
        "check": "integration gate",
        "passed": bool(checks["passed"].all()),
        "details": f"{int(checks['passed'].sum())}/{len(checks)}",
    },
    {
        "check": "common population",
        "passed": len(common_manifest) == 374,
        "details": str(common_manifest["split"].value_counts().to_dict()),
    },
    {
        "check": "selected n remains frozen",
        "passed": selected_n == EXPECTED_SELECTED_N,
        "details": f"n={selected_n}",
    },
    {
        "check": "feature contracts",
        "passed": (
            len(feature_sets["P1_TOTAL_SELECTED"]) == 6
            and len(feature_sets["P1_HOMEAWAY_SELECTED"]) == 18
            and len(feature_sets["MD1_BASE_COMMON"]) == 128
            and len(feature_sets["MD1_PLUS_P1_TOTAL"]) == 134
            and len(feature_sets["MD1_PLUS_P1_HOMEAWAY"]) == 146
        ),
        "details": str({k: len(v) for k, v in feature_sets.items()}),
    },
    {
        "check": "validation decision frozen before test",
        "passed": (EXPERIMENT_ROOT / "audit" / "FROZEN_BEFORE_TEST.json").exists(),
        "details": str(EXPERIMENT_ROOT / "audit" / "FROZEN_BEFORE_TEST.json"),
    },
    {
        "check": "validation populations identical",
        "passed": len(val_predictions) == 77 * 7,
        "details": f"prediction rows={len(val_predictions)}",
    },
]

if RUN_TEST_STAGE:
    readiness_rows += [
        {
            "check": "test populations identical",
            "passed": len(test_predictions) == 46 * 7,
            "details": f"prediction rows={len(test_predictions)}",
        },
        {
            "check": "paired bootstrap generated",
            "passed": len(bootstrap) == 8,
            "details": f"comparisons={len(bootstrap)}",
        },
    ]

exp_readiness = pd.DataFrame(readiness_rows)
exp_readiness.to_csv(
    EXPERIMENT_ROOT / "audit" / "exp01_readiness.csv", index=False
)

if not bool(exp_readiness["passed"].all()):
    display(exp_readiness.loc[~exp_readiness["passed"]])
    raise AssertionError("EXP01 readiness failed.")

summary = {
    "experiment": "EXP01_P1_REPRESENTATION_TASK_C",
    "status": "PASS",
    "selected_n": selected_n,
    "fixed_n": fixed_n,
    "common_matches": len(common_manifest),
    "common_train": int((common_manifest["split"] == "train").sum()),
    "common_validation": int((common_manifest["split"] == "validation").sum()),
    "common_test": int((common_manifest["split"] == "test").sum()),
    "representation_feature_counts": {k: len(v) for k, v in feature_sets.items()},
    "validation_decision": decision,
    "test_stage_run": bool(RUN_TEST_STAGE),
    "validation_metrics": val_metrics.to_dict("records"),
    "test_metrics": test_metrics.to_dict("records") if RUN_TEST_STAGE else None,
    "paired_bootstrap": bootstrap.to_dict("records") if RUN_TEST_STAGE else None,
}
write_json(summary, EXPERIMENT_ROOT / "audit" / "experiment_summary.json")

# Human-readable experiment README.
readme_lines = [
    "# EXP01 — P1 Representation Ablation, Task C",
    "",
    "This directory is generated from frozen upstream MD1 and Berrar P1 outputs.",
    "",
    f"- Berrar selected recency: n={selected_n}",
    f"- Fixed-recency comparator: n={fixed_n}",
    f"- Common population: {len(common_manifest)} matches",
    f"- Split: {summary['common_train']} train / {summary['common_validation']} validation / {summary['common_test']} test",
    "- Model probe: one fixed Random Forest configuration for every learned representation",
    "- Primary metric: RPS",
    "- Representation selection: validation RPS only",
    "- Test: report only",
    "",
    "See `audit/FROZEN_BEFORE_TEST.json` for the pre-test freeze record.",
    "See `metrics/` for validation/test metrics and paired bootstrap results.",
]
(EXPERIMENT_ROOT / "README.md").write_text("\n".join(readme_lines) + "\n", encoding="utf-8")

print("EXP01 READINESS")
display(exp_readiness)

print("Validation metrics")
display(val_metrics)

if RUN_TEST_STAGE:
    print("Test metrics")
    display(test_metrics)
    print("Paired bootstrap")
    display(bootstrap)

print()
print("EXP01 COMPLETE:", EXPERIMENT_ROOT)


EXP01 READINESS


,check,passed,details
0,upstream MD1 readiness,True,15/15
1,upstream P1 readiness,True,15/15
2,integration gate,True,20/20
3,common population,True,"{'train': 251, 'validation': 77, 'test': 46}"
4,selected n remains frozen,True,n=28
5,feature contracts,True,"{'P1_TOTAL_FIXED': 6, 'P1_TOTAL_SELECTED': 6, ..."
6,validation decision frozen before test,True,/content/drive/MyDrive/ml_project/code/data_pi...
7,validation populations identical,True,prediction rows=539
8,test populations identical,True,prediction rows=322
9,paired bootstrap generated,True,comparisons=8


Validation metrics


,representation,n_matches,rps,log_loss,brier,accuracy,ece,feature_count,model
0,MARKET,77,0.196931,0.974603,0.580026,0.545455,0.060403,0,De-vigged Football-Data 1X2 market baseline
1,P1_TOTAL_SELECTED,77,0.212926,1.039698,0.618526,0.441558,0.107637,6,RandomForest fixed probe
2,P1_HOMEAWAY_SELECTED,77,0.213131,1.038484,0.619191,0.467532,0.069547,18,RandomForest fixed probe
3,MD1_PLUS_P1_HOMEAWAY,77,0.217547,1.048381,0.626288,0.467532,0.078942,146,RandomForest fixed probe
4,MD1_PLUS_P1_TOTAL,77,0.218368,1.041537,0.622437,0.467532,0.103201,134,RandomForest fixed probe
5,P1_TOTAL_FIXED,77,0.223466,1.051925,0.634180,0.506494,0.106890,6,RandomForest fixed probe
6,MD1_BASE_COMMON,77,0.223629,1.064525,0.639151,0.441558,0.036884,128,RandomForest fixed probe


Test metrics


,representation,n_matches,rps,log_loss,brier,accuracy,ece,feature_count,model
0,MARKET,46,0.174848,0.961138,0.573607,0.521739,0.141250,0,De-vigged Football-Data 1X2 market baseline
1,MD1_BASE_COMMON,46,0.197268,1.016214,0.605964,0.456522,0.121464,128,RandomForest fixed probe
2,MD1_PLUS_P1_TOTAL,46,0.200219,1.023407,0.612775,0.413043,0.098638,134,RandomForest fixed probe
3,MD1_PLUS_P1_HOMEAWAY,46,0.202090,1.042429,0.622532,0.456522,0.066491,146,RandomForest fixed probe
4,P1_TOTAL_FIXED,46,0.208305,1.046362,0.623790,0.521739,0.095419,6,RandomForest fixed probe
5,P1_HOMEAWAY_SELECTED,46,0.220792,1.133443,0.685315,0.369565,0.136783,18,RandomForest fixed probe
6,P1_TOTAL_SELECTED,46,0.229433,1.167425,0.700290,0.456522,0.168881,6,RandomForest fixed probe


Paired bootstrap


,left,right,n_matches,delta_definition,mean_delta_rps,ci95_low,ci95_high,bootstrap_resamples,bootstrap_seed,fraction_bootstrap_delta_lt_0
0,P1_TOTAL_SELECTED,P1_TOTAL_FIXED,46,RPS(left) - RPS(right); negative favors left,0.021128,-0.012849,0.056051,10000,42,0.1119
1,P1_HOMEAWAY_SELECTED,P1_TOTAL_SELECTED,46,RPS(left) - RPS(right); negative favors left,-0.008641,-0.027178,0.009187,10000,42,0.8233
2,P1_HOMEAWAY_SELECTED,MD1_BASE_COMMON,46,RPS(left) - RPS(right); negative favors left,0.023524,-0.003706,0.049107,10000,42,0.0440
3,MD1_PLUS_P1_TOTAL,MD1_BASE_COMMON,46,RPS(left) - RPS(right); negative favors left,0.002951,-0.003778,0.009444,10000,42,0.1823
4,MD1_PLUS_P1_HOMEAWAY,MD1_BASE_COMMON,46,RPS(left) - RPS(right); negative favors left,0.004822,-0.003589,0.013242,10000,42,0.1321
5,MD1_BASE_COMMON,MARKET,46,RPS(left) - RPS(right); negative favors left,0.022420,-0.003951,0.048824,10000,42,0.0464
6,MD1_PLUS_P1_TOTAL,MARKET,46,RPS(left) - RPS(right); negative favors left,0.025372,0.001179,0.049757,10000,42,0.0198
7,MD1_PLUS_P1_HOMEAWAY,MARKET,46,RPS(left) - RPS(right); negative favors left,0.027242,0.002782,0.051309,10000,42,0.0136



EXP01 COMPLETE: /content/drive/MyDrive/ml_project/code/data_pipeline/results/EXP01_P1_REPRESENTATION_TASK_C



## Interpretation protocol

Use the saved outputs rather than manually selecting the best test row.

The report should answer these controlled comparisons:

- **H1 Recency:** `P1_TOTAL_SELECTED` vs `P1_TOTAL_FIXED`
- **H2 Venue representation:** `P1_HOMEAWAY_SELECTED` vs `P1_TOTAL_SELECTED`
- **H3 Compact P1 vs rich MD1:** `P1_HOMEAWAY_SELECTED` vs `MD1_BASE_COMMON`
- **H4 Incremental P1 value:** `MD1_PLUS_P1_HOMEAWAY` and `MD1_PLUS_P1_TOTAL` vs `MD1_BASE_COMMON`
- **Market:** compare models and de-vigged market on the exact same 46 common test matches

For paired bootstrap rows, the stored delta is:

```text
RPS(left) - RPS(right)
```

so a **negative** value favors the left representation.

Do not re-select Berrar `n` after seeing EXP01 validation or test results.
